# Week 4: Unit test generator

In [ ]:
import os
import sys
import tempfile
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import display

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "pytest"],
    check=True,
    text=True,
    capture_output=True,
)

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key and openrouter_api_key.startswith('sk-proj-') and len(openrouter_api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

In [ ]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
client = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL) if openrouter_api_key else None

models = [
    "openai/gpt-4o-mini",
    "qwen/qwen-2.5-coder-32b-instruct",
    "mistralai/codestral-2508",
]

test_frameworks = ["pytest (Python)", "unittest (Python)", "Jest (JavaScript/TS)", "Go testing", "JUnit (Java)"]

In [ ]:
def strip_code_fences(text):
    for tag in ("```python", "```py", "```javascript", "```js", "```go", "```java", "```"):
        text = text.replace(tag, "")
    return text.strip()

system_prompt = """Generate unit tests only. Output nothing but test code. No explanation. Assume the code under test is in module 'src' (saved as src.py); in tests use: from src import ... or import src. Infer behavior only from the actual code; do not assume semantics from function or variable names."""

def user_prompt_for(code, framework):
    return f"""Use {framework}. Cover main behavior and edge cases. Output only the test code. Test only observable behavior: do not assert that an exception is raised unless the code actually raises it for that input. Only assert exceptions or edge-case behavior that the code actually exhibits; do not invent edge cases (e.g. wrong-type inputs raising errors) when the implementation allows them.

Code under test (will be saved as src.py):
```python
{code}
```"""

def messages_for(code, framework):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(code, framework)},
    ]

def generate(model_id, messages, max_tokens=2048, temperature=0.2):
    kwargs = dict(model=model_id, messages=messages, max_tokens=max_tokens, temperature=temperature)
    if "gpt" in model_id:
        kwargs["reasoning_effort"] = "medium"
    r = client.chat.completions.create(**kwargs)
    return r.choices[0].message.content or ""

In [ ]:
def run(model_id, framework, user_input):
    if not client:
        return "# Set OPENROUTER_API_KEY in .env"
    if not user_input or not user_input.strip():
        return ""
    reply = generate(model_id, messages_for(user_input, framework))
    return strip_code_fences(reply)

def run_tests(source_code, test_code):
    if not source_code or not test_code:
        return "Paste code and generate tests first."
    try:
        with tempfile.TemporaryDirectory() as d:
            with open(os.path.join(d, "src.py"), "w") as f:
                f.write(source_code)
            with open(os.path.join(d, "test_src.py"), "w") as f:
                f.write(strip_code_fences(test_code))
            r = subprocess.run(
                [sys.executable, "-m", "pytest", "test_src.py", "-v"],
                cwd=d,
                capture_output=True,
                text=True,
                timeout=30,
            )
            err = r.stderr or ""
            if "No module named pytest" in err or "No module named 'pytest'" in err:
                return "pytest not installed. In a notebook cell run: pip install pytest"
            out = (r.stdout or "") + err
            return out if out else ("OK" if r.returncode == 0 else "Tests failed.")
    except subprocess.TimeoutExpired:
        return "Run timed out."
    except Exception as e:
        return str(e)

with gr.Blocks() as demo:
    gr.Markdown("### Unit test generator")
    code_in = gr.Code(label="Code", value="def add(a, b):\n    return a + b", language="python", lines=14)
    tests_out = gr.Code(label="Generated tests", value="", language="python", lines=14)
    with gr.Row():
        model = gr.Dropdown(models, value=models[0], label="Model")
        framework = gr.Dropdown(test_frameworks, value=test_frameworks[0], label="Framework")
    gen_btn = gr.Button("Generate")
    run_out = gr.Textbox(label="Run tests (pytest)", lines=8)
    run_btn = gr.Button("Run tests")
    gen_btn.click(fn=run, inputs=[model, framework, code_in], outputs=tests_out)
    run_btn.click(fn=run_tests, inputs=[code_in, tests_out], outputs=run_out)

demo.launch()